# 03. Biến đổi dữ liệu, gom cụm và so sánh các phương án

**Học phần:** Khai thác dữ liệu — Nhóm 12

Notebook này chuẩn hóa ma trận đặc trưng, khảo sát số cụm cho từng năm, chạy năm cấu hình gom cụm
trên cả năm năm 2015-2019 và so sánh kết quả bằng các độ đo nội tại.

Toàn bộ thuật toán và độ đo được gọi từ `src/models/clustering.py` do nhóm tự cài đặt; notebook không
dùng thư viện học máy nào. Kết quả được ghi ra `reports/tables/`, `reports/figures/`, `models/` và
nhãn cụm của từng quốc gia ở `data/processed/per_year/` để notebook 04 tiếp tục xử lý.

In [ ]:
from pathlib import Path
import sys

# Dò ngược lên để tìm gốc repo (thư mục chứa 'src')
here = Path.cwd().resolve()
project_root = next(p for p in [here, *here.parents] if (p / 'src').is_dir())
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

figures_dir = project_root / 'reports' / 'figures'
tables_dir = project_root / 'reports' / 'tables'
per_year_dir = project_root / 'data' / 'processed' / 'per_year'
models_dir = project_root / 'models'
for folder in (figures_dir, tables_dir, per_year_dir, models_dir):
    folder.mkdir(parents=True, exist_ok=True)

print('Gốc repo:', project_root)

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()
project_root = next((p for p in [here, *here.parents] if (p / 'src').is_dir()), Path('..').resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.data_loader import load_interim
from src.features.feature_engineering import FEATURE_COLUMNS, scale_features
from src.models.clustering import (
    calculate_adjusted_rand_index,
    evaluate_clustering,
    find_optimal_k_kmeans,
    run_dbscan,
    run_hierarchical,
    run_kmeans,
    save_cluster_artifacts,
)
from src.visualization.plots import (
    plot_dendrogram,
    plot_elbow_silhouette_by_year,
    plot_heatmap,
    plot_scaling_comparison,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

YEARS = [2015, 2016, 2017, 2018, 2019]
K_RANGE = list(range(2, 10))
EPS_VALUES = [0.6, 0.8, 1.0, 1.2]
MIN_SAMPLES_VALUES = [3, 4, 5]
SEEDS = list(range(10))
N_INIT = 10
MAIN_YEAR = 2019
DBSCAN_CORE = {'eps': 1.0, 'min_samples': 4}
print('Cấu hình thực nghiệm:', YEARS, '| k khảo sát:', K_RANGE)

## 1. Biến đổi dữ liệu sang biểu diễn phù hợp với thuật toán gom cụm

K-Means và phân cấp Ward đều dựa trên khoảng cách Euclid, nên nếu không chuẩn hóa thì đặc trưng có
độ lệch lớn (GDP per capita) sẽ chi phối kết quả. Vì vậy mỗi năm được chuẩn bị ba phiên bản dữ liệu:
Z-score, Min-Max và giữ nguyên giá trị gốc để làm phương án đối chứng.

In [ ]:
interim = load_interim()
print('Bảng trung gian:', interim.shape)

data_by_year = {}
for year in YEARS:
    frame = interim[interim['year'] == year].dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
    X_standard, _ = scale_features(frame, method='standard')
    X_minmax, _ = scale_features(frame, method='minmax')
    X_raw = frame[FEATURE_COLUMNS].values.astype(float)
    data_by_year[year] = {'frame': frame, 'standard': X_standard, 'minmax': X_minmax, 'none': X_raw}
    print(f'Năm {year}: {len(frame)} quốc gia (đã loại dòng thiếu dữ liệu ở 6 yếu tố cơ sở)')

## 2. Khảo sát số cụm cho từng năm

Với mỗi năm, chạy K-Means cho k từ 2 đến 9 và ghi lại inertia (WCSS) cùng hệ số Silhouette.
Số cụm tối ưu của mỗi năm là giá trị k cho Silhouette lớn nhất (nếu bằng nhau thì chọn k nhỏ hơn).

In [ ]:
metrics_by_year = {}
sensitivity_rows = []
start = time.time()

for year in YEARS:
    results = find_optimal_k_kmeans(data_by_year[year]['standard'], k_range=K_RANGE, n_init=N_INIT, random_state=42)
    table = pd.DataFrame(results)
    table.insert(0, 'year', year)
    metrics_by_year[year] = table
    sensitivity_rows.append(table)

sensitivity_df = pd.concat(sensitivity_rows, ignore_index=True)
sensitivity_df.to_csv(tables_dir / 'k_sensitivity_by_year.csv', index=False, encoding='utf-8-sig')

optimal_k = {}
for year, table in metrics_by_year.items():
    best = table.sort_values(['silhouette', 'k'], ascending=[False, True]).iloc[0]
    optimal_k[year] = int(best['k'])

print(f'Thời gian khảo sát số cụm: {time.time() - start:.1f} giây')
print('Số cụm tối ưu từng năm:', optimal_k)
sensitivity_df.head(8)

In [ ]:
plot_elbow_silhouette_by_year(metrics_by_year, save_path=str(figures_dir / 'elbow_silhouette_by_year.png'))
plt.show()
print('Đã lưu:', figures_dir / 'elbow_silhouette_by_year.png')

## 3. So sánh năm cấu hình gom cụm trên cả năm năm

Năm cấu hình lõi: K-Means với Z-score, K-Means với Min-Max, K-Means không chuẩn hóa, phân cấp Ward
với Z-score và DBSCAN với Z-score. K-Means và Ward chạy với số cụm tối ưu của từng năm; DBSCAN dùng
`eps = 1.0`, `min_samples = 4` (phần sau sẽ khảo sát độ nhạy hai tham số này).

In [ ]:
CORE_CONFIGURATIONS = [
    ('K-Means (Z-score)', 'kmeans', 'standard'),
    ('K-Means (Min-Max)', 'kmeans', 'minmax'),
    ('K-Means (không chuẩn hóa)', 'kmeans', 'none'),
    ('Phân cấp Ward (Z-score)', 'hierarchical', 'standard'),
    ('DBSCAN (Z-score)', 'dbscan', 'standard'),
]

comparison_rows = []
labels_by_year = {}
start = time.time()

for year in YEARS:
    k = optimal_k[year]
    labels_by_year[year] = {}
    for name, method, scaling in CORE_CONFIGURATIONS:
        X = data_by_year[year][scaling]
        if method == 'kmeans':
            labels, model = run_kmeans(X, n_clusters=k, n_init=N_INIT, random_state=42)
        elif method == 'hierarchical':
            labels, model = run_hierarchical(X, n_clusters=k, linkage='ward')
        else:
            labels, model = run_dbscan(X, eps=DBSCAN_CORE['eps'], min_samples=DBSCAN_CORE['min_samples'])
        labels_by_year[year][name] = labels
        row = {'year': year, 'configuration': name, 'k_input': k, **evaluate_clustering(X, labels)}
        comparison_rows.append(row)

        metrics = {key: value for key, value in row.items() if key not in ('year', 'configuration')}
        if name == 'K-Means (Z-score)':
            save_cluster_artifacts(models_dir / f'kmeans_{year}.json', year=year, method='K-Means',
                                   scaler='standard', n_clusters=k, centroids=model.centroids,
                                   feature_columns=FEATURE_COLUMNS, metrics=metrics)
        elif name == 'Phân cấp Ward (Z-score)':
            save_cluster_artifacts(models_dir / f'hierarchical_{year}.json', year=year, method='Ward',
                                   scaler='standard', n_clusters=k, centroids=model.cluster_centroids,
                                   feature_columns=FEATURE_COLUMNS, metrics=metrics)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(tables_dir / 'model_comparison_metrics_by_year.csv', index=False, encoding='utf-8-sig')
print(f'Thời gian chạy ma trận cấu hình: {time.time() - start:.1f} giây')
comparison_df

In [ ]:
plot_scaling_comparison(comparison_df, metric='silhouette', save_path=str(figures_dir / 'scaling_comparison.png'))
plt.show()

print('Silhouette trung bình theo cấu hình (trên 5 năm):')
print(comparison_df.groupby('configuration')['silhouette'].mean().sort_values(ascending=False).round(4))

## 4. Độ nhạy tham số của DBSCAN

Quét `eps` và `min_samples` để xem số cụm và số điểm bị coi là nhiễu thay đổi thế nào. DBSCAN không
cần khai báo trước số cụm nhưng rất nhạy với hai tham số này.

In [ ]:
sweep_rows = []
start = time.time()
for year in YEARS:
    X = data_by_year[year]['standard']
    for eps in EPS_VALUES:
        for min_samples in MIN_SAMPLES_VALUES:
            labels, model = run_dbscan(X, eps=eps, min_samples=min_samples)
            sweep_rows.append({'year': year, 'eps': eps, 'min_samples': min_samples,
                               'n_clusters': model.n_clusters, 'n_noise': model.n_noise})

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(tables_dir / 'dbscan_sweep.csv', index=False, encoding='utf-8-sig')
print(f'Thời gian quét DBSCAN: {time.time() - start:.1f} giây | số cấu hình: {len(sweep_df)}')

fig, axes = plt.subplots(1, len(YEARS), figsize=(4 * len(YEARS), 3.6))
for axis, year in zip(axes, YEARS):
    pivot = sweep_df[sweep_df['year'] == year].pivot(index='min_samples', columns='eps', values='n_clusters')
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu', ax=axis, cbar=False)
    axis.set_title(f'Năm {year}', fontsize=11)
    axis.set_xlabel('eps')
    axis.set_ylabel('min_samples' if year == YEARS[0] else '')
fig.suptitle('Số cụm DBSCAN theo eps và min_samples (dữ liệu Z-score)', fontsize=13)
plt.tight_layout()
fig.savefig(figures_dir / 'dbscan_sweep_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
sweep_df.head(12)

## 5. Độ ổn định của nhãn cụm

K-Means phụ thuộc điểm khởi tạo, nên chạy lại 10 lần với các seed khác nhau rồi đo mức trùng khớp
nhãn bằng Adjusted Rand Index. Phân cấp Ward và DBSCAN là thuật toán tất định nên chỉ số này bằng 1.

In [ ]:
stability_rows = []
stability_configs = [
    ('K-Means (Z-score)', 'standard'),
    ('K-Means (Min-Max)', 'minmax'),
    ('K-Means (không chuẩn hóa)', 'none'),
]
start = time.time()


def stability_across_seeds(X, k, n_init):
    """Chạy K-Means với 10 seed khác nhau; trả về ARI trung bình và nhỏ nhất so với lần chạy đầu."""
    reference, _ = run_kmeans(X, n_clusters=k, n_init=n_init, random_state=SEEDS[0])
    scores = []
    for seed in SEEDS[1:]:
        labels, _ = run_kmeans(X, n_clusters=k, n_init=n_init, random_state=seed)
        scores.append(calculate_adjusted_rand_index(reference, labels))
    return round(float(np.mean(scores)), 4), round(float(np.min(scores)), 4)


for year in YEARS:
    k = optimal_k[year]
    for name, scaling in stability_configs:
        X = data_by_year[year][scaling]
        # n_init = 1 đo độ nhạy với điểm khởi tạo; n_init = 10 là cấu hình dùng trong báo cáo
        mean_single, min_single = stability_across_seeds(X, k, n_init=1)
        mean_multi, min_multi = stability_across_seeds(X, k, n_init=N_INIT)
        stability_rows.append({'year': year, 'configuration': name,
                               'mean_ari_single_init': mean_single, 'min_ari_single_init': min_single,
                               'mean_ari_n_init_10': mean_multi, 'min_ari_n_init_10': min_multi})
    for deterministic in ['Phân cấp Ward (Z-score)', 'DBSCAN (Z-score)']:
        stability_rows.append({'year': year, 'configuration': deterministic,
                               'mean_ari_single_init': 1.0, 'min_ari_single_init': 1.0,
                               'mean_ari_n_init_10': 1.0, 'min_ari_n_init_10': 1.0})

stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv(tables_dir / 'stability_ari.csv', index=False, encoding='utf-8-sig')
print(f'Thời gian đo độ ổn định: {time.time() - start:.1f} giây')

ari_matrix = (stability_df[stability_df['configuration'].str.startswith('K-Means')]
              .pivot(index='configuration', columns='year', values='mean_ari_single_init'))
plot_heatmap(ari_matrix, 'Độ ổn định khi chỉ chạy một lần khởi tạo (ARI trung bình trên 10 seed)',
             'Năm', 'Cấu hình', save_path=str(figures_dir / 'stability_ari_heatmap.png'), fmt='.3f')
plt.show()
stability_df

## 6. Cấu hình đối chứng ba cụm

Vì số cụm tối ưu có thể khác nhau giữa các năm nên nhãn cụm không so sánh trực tiếp được. Cấu hình
ba cụm với Z-score được chạy cho mọi năm để làm mốc so sánh xuyên năm ở notebook 04.

In [ ]:
contrast_rows = []
for year in YEARS:
    X = data_by_year[year]['standard']
    labels, model = run_kmeans(X, n_clusters=3, n_init=N_INIT, random_state=42)
    contrast_rows.append({'year': year, 'k_input': 3, **evaluate_clustering(X, labels)})
    frame = data_by_year[year]['frame'][['country']].copy()
    frame.insert(0, 'year', year)
    frame['cluster_k3'] = labels
    frame.to_csv(per_year_dir / f'contrast_k3_{year}.csv', index=False, encoding='utf-8-sig')

contrast_df = pd.DataFrame(contrast_rows)
contrast_df.to_csv(tables_dir / 'contrast_k3_by_year.csv', index=False, encoding='utf-8-sig')
contrast_df

## 7. Lưu nhãn cụm của từng quốc gia và biểu đồ cây

Nhãn cụm của cả năm cấu hình được lưu theo từng năm để notebook 04 dùng lại, tránh phải chạy lại
thuật toán. Biểu đồ cây chỉ vẽ cho năm 2019 để minh họa quá trình hợp nhất của Ward.

In [ ]:
for year in YEARS:
    frame = data_by_year[year]['frame'][['country']].copy()
    frame.insert(0, 'year', year)
    frame['cluster_kmeans_standard'] = labels_by_year[year]['K-Means (Z-score)']
    frame['cluster_kmeans_minmax'] = labels_by_year[year]['K-Means (Min-Max)']
    frame['cluster_kmeans_raw'] = labels_by_year[year]['K-Means (không chuẩn hóa)']
    frame['cluster_ward'] = labels_by_year[year]['Phân cấp Ward (Z-score)']
    frame['cluster_dbscan'] = labels_by_year[year]['DBSCAN (Z-score)']
    frame.to_csv(per_year_dir / f'clustering_{year}.csv', index=False, encoding='utf-8-sig')

print('Đã lưu nhãn cụm từng năm vào', per_year_dir)
print('Đã lưu tham số mô hình vào', models_dir)

In [ ]:
X_main = data_by_year[MAIN_YEAR]['standard']
k_main = optimal_k[MAIN_YEAR]
ward_labels, ward_model = run_hierarchical(X_main, n_clusters=k_main, linkage='ward')

plot_dendrogram(ward_model.merge_history, n_clusters=k_main,
                title=f'Biểu đồ cây phân cấp Ward - năm {MAIN_YEAR}',
                save_path=str(figures_dir / f'dendrogram_ward_{MAIN_YEAR}.png'))
plt.show()

print(f'Cắt cây ở {k_main} cụm cho năm {MAIN_YEAR}.')
print('Số quốc gia mỗi cụm:', pd.Series(ward_labels).value_counts().sort_index().tolist())
print('Độ cao hợp nhất lớn nhất:', round(ward_model.merge_history[-1]['height'], 3))

## 8. Tổng kết notebook 03

- Số cụm tối ưu được chọn riêng cho từng năm theo hệ số Silhouette.
- Năm cấu hình gom cụm đã được chạy trên cả năm năm và đánh giá bằng bốn độ đo.
- DBSCAN được khảo sát độ nhạy với 12 tổ hợp `eps` và `min_samples` mỗi năm.
- Độ ổn định của K-Means được đo trên 10 seed.

Kết quả chi tiết:
- `reports/tables/k_sensitivity_by_year.csv`
- `reports/tables/model_comparison_metrics_by_year.csv`
- `reports/tables/dbscan_sweep.csv`
- `reports/tables/stability_ari.csv`
- `reports/tables/contrast_k3_by_year.csv`
- Hình: `elbow_silhouette_by_year.png`, `scaling_comparison.png`, `dbscan_sweep_heatmap.png`,
  `stability_ari_heatmap.png`, `dendrogram_ward_2019.png`

**Notebook tiếp theo:** `04_cluster_profiling_and_evaluation.ipynb` — chân dung từng cụm, ghép cụm
giữa các năm, bảng dịch chuyển cụm và hậu kiểm bằng điểm hạnh phúc.